# Plot Constructor Example

This notebook demonstrates how to build stacked plots from processor results.

In [ ]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd

from app.visualization import PlotConstructor

## 1) Load/prepare processor results

Below is an example of loading data from existing processors if files are already downloaded.
If files are unavailable, the notebook builds a small synthetic fallback dataset so plotting cells still run.

In [ ]:
from app.gfz.gfz_processor import GfzProcessor
from app.kyoto.kyoto_dst_processor import KyotoProcessor
from app.omni.omni_processor import OmniProcessor
from app.simurg.simurg_processor import SimurgProcessor, DataProduct

date_str = "2025-11-12"
base_dir = Path.cwd().parent
download_dir = base_dir / "files" / date_str

kp_df = GfzProcessor(str(download_dir / "kp")).load(date_str=date_str)
dst_df = KyotoProcessor(str(download_dir / "dst")).load(date_str=date_str)
omni_df = OmniProcessor(str(download_dir / "omni")).load(date_str=date_str)
roti_map = SimurgProcessor(str(download_dir / "simurg")).load(date_str, product_type=DataProduct.ROTI)

print("Loaded:")
print("  kp:", None if kp_df is None else kp_df.shape)
print("  dst:", None if dst_df is None else dst_df.shape)
print("  omni:", None if omni_df is None else omni_df.shape)
print("  roti_map:", None if roti_map is None else len(roti_map))

In [ ]:
# Fallback synthetic data for reproducible demo
if kp_df is None:
    kp_df = pd.DataFrame({"datetime": pd.date_range("2025-11-12", periods=24, freq="1h"), "kp": np.random.randint(0, 9, 24)})

if dst_df is None:
    dst_df = pd.DataFrame({"datetime": pd.date_range("2025-11-12", periods=24, freq="1h"), "dst": np.random.normal(-30, 20, 24)})

if omni_df is None:
    omni_df = pd.DataFrame({"DateTime": pd.date_range("2025-11-12", periods=24, freq="1h"), "bz": np.random.normal(0, 4, 24), "symh": np.random.normal(-20, 15, 24)})

if roti_map is None:
    ts = datetime(2025, 11, 12, 0, 0, tzinfo=timezone.utc)
    lats = np.linspace(-80, 80, 24)
    lons = np.linspace(-180, 180, 36)
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    vals = np.abs(np.sin(np.radians(lat_grid)))
    arr = np.array(list(zip(lat_grid.ravel(), lon_grid.ravel(), vals.ravel())), dtype=[("lat", "f8"), ("lon", "f8"), ("vals", "f8")])
    roti_map = {ts: arr}

In [ ]:
processor_results = {
    "ROTI": roti_map,
    "Kp": kp_df,
    "Dst": dst_df,
    "OMNI": omni_df,
}

plotter = PlotConstructor(processor_results)

## 2) Inspect available plot names

In [ ]:
plotter.available_plots()

## 3) Build stacked plots (order is preserved)

In [ ]:
fig, _ = plotter.plot(["ROTI", "Dst", "Kp"])
fig

## 4) Change order of subplots

In [ ]:
fig, _ = plotter.plot(["Kp", "Dst", "ROTI"])
fig

## 5) Pass per-plot parameters

In [ ]:
first_roti_time = sorted(roti_map.keys())[0]

fig, _ = plotter.plot([
    {"name": "ROTI", "params": {"plot_time": first_roti_time, "cmap": "viridis", "s": 10}},
    {"name": "Kp", "params": {"bins": 9, "color": "orange"}},
    {"name": "Dst", "params": {"color": "black"}},
])
fig